# SERTS – wie funktioniert die Berechnung?

Dieses Notebook geht den Hauptcode Schritt für Schritt durch. Es enthält **keine eigene Logik** –
alles kommt aus dem Paket `src/serts`. Kernel in VS Code: `.venv` des Repos (oben rechts „Select Kernel“).

**Ablauf eines Laufs** (`uv run serts run` macht genau das):

```
config/base.yaml ──> 1 Zeitreihen (prepare.py) ──> 2 Netz bauen (network.py)
                                                    3 Zusatzbedingungen (constraints.py)
                                                    4 Optimieren (PyPSA + HiGHS)
                     6 Grafiken (ergebnisplots.py) <── 5 Kennzahlen + Kosten (kpi.py, economics.py)
```

In [ ]:
%matplotlib inline
import logging, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from serts.config import load_config
from serts import network, kpi, economics, ergebnisplots
from serts.prepare import build_zeitreihen
from serts.run import loese

logging.getLogger("linopy").setLevel(logging.WARNING)
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("serts").setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 80)

## 0 Konfiguration

Alle Zahlen stehen in `config/base.yaml`. Jeder Parameter hat `wert`, `einheit`, `status`
(belegt / vorgabe / annahme / offen) und `quelle`. `cfg.get("pfad")` liefert den Wert und bricht ab,
wenn er noch offen (`null`) ist – es gibt keine stillen Standardwerte.

**Netzfälle:** Dieses Notebook rechnet mit dem **kleinen Netz** (Basis). Das größere Netz lädt man mit
`load_config(netz="gross")` bzw. `uv run serts run --netz gross` (Ausgaben getrennt in `results/gross/`).
Die festen Leistungen und Größen in den Abschnitten 2–4 sind auf das kleine Netz zugeschnitten.
Eine **zentrale Übersicht der wichtigsten Eingangsparameter** (beide Netze nebeneinander, mit Status und
Quelle) steht unten und als Datei `docs/eingangsparameter.md` (`uv run serts uebersicht`).

In [ ]:
cfg = load_config()                      # oder load_config("wetter_2022") für ein Szenario
tabelle = pd.DataFrame(
    [(p, d["wert"], d["einheit"], d["status"]) for p, d in cfg.params()],
    columns=["Parameter", "Wert", "Einheit", "Status"],
)
tabelle[tabelle["Parameter"].str.startswith(("ptes.", "kollektor.eta", "strom.preis", "oekonomie.zins"))]

from IPython.display import Markdown
from serts.uebersicht import uebersicht_markdown
display(Markdown(uebersicht_markdown()))   # Netz klein | groß, Stand der Config


## 1 Zeitreihen (8760 Stunden)

`build_zeitreihen` sammelt alles, was sich stündlich ändert:

| Spalte | Bedeutung | Herkunft |
|---|---|---|
| `last_gesamt` | Wärmebedarf am Netz inkl. Netzverlusten [MW] | Netzberechnung Bad Lippspringe 2022 |
| `T_a` | Außentemperatur [°C] | DWD-Station 03028, 2022 |
| `COP_luft` | COP der Luft-WP (KWW-Ansatz: Gütegrad 45 % × Carnot auf den Vorlauf) | `heatpump.py` |
| `G_poa`, `P_solar` | Strahlung auf Kollektorebene [W/m²], nutzbare Kollektorleistung [kW/m²] | PVGIS-Normaljahr, `solar.py` |
| `strompreis` | Strompreis [€/MWh] = Energie + Netzentgelt + Abgaben | `prices.py`: fest 30 ct/kWh **oder** Day-Ahead-Zeitreihe (Abschnitt 7) |

In [ ]:
ts, _ = build_zeitreihen(cfg)
print(f"Wärmebedarf: {ts['last_gesamt'].sum():,.0f} MWh/a, Spitze {ts['last_gesamt'].max():.2f} MW")
print(f"Kollektor:   {ts['P_solar'].sum():,.0f} kWh je m² und Jahr nutzbar")
print(f"COP Luft:    {ts['COP_luft'].min():.2f} … {ts['COP_luft'].max():.2f}")

fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ts["last_gesamt"].resample("D").mean().plot(ax=ax[0], title="Last [MW], Tagesmittel")
ts["P_solar"].resample("D").sum().plot(ax=ax[1], title="Kollektorertrag [kWh/m² je Tag]", color="orange")
ts["COP_luft"].resample("D").mean().plot(ax=ax[2], title="COP Luft-WP, Tagesmittel", color="green")
plt.tight_layout()
plt.show()

## 2 Das Netz (PyPSA)

PyPSA beschreibt das System als **Knoten (Busse)** und **Komponenten**, die Energie zwischen ihnen
verschieben. Die Optimierung wählt für jede Stunde, wie viel jede Komponente liefert, und (falls
gewünscht) wie groß sie gebaut wird – so, dass die Jahreskosten minimal sind.

```
                 ┌──────────── solar_direkt ─────────────┐
    Kollektor ──>│ solar                                 ▼
                 └── laden ──> [Speicher heiß | kalt] ──> entladen_direkt ──> waerme ──> Last
                                  (53 % | 47 %)  └── Quellwärme ──┐              ▲   ▲
    Netzstrom ──> hp_luft ─────────────────────────────────> wp ──┴> wp_out ─────┘   │
             └──> hp_speicher (holt Quellwärme aus dem kalten Teil) ┘                 │
             └──> pth ──────────────────────────────────────────────────────────────┘
```

- **Generator**: Quelle (Netzstrom zum Strompreis; Kollektor mit stündlicher Verfügbarkeit `P_solar`).
- **Link**: Umwandlung von `bus0` nach `bus1` mit Wirkungsgrad. `laden` hat zwei Ausgänge (heiß und kalt),
  `hp_speicher` entnimmt (COP − 1) MWh Quellwärme je MWh Strom aus dem kalten Speicherteil.
- **Store**: der Erdbeckenspeicher in **zwei Teilen**: *heiß* (Wärme zwischen 85 °C und dem Netz-Rücklauf,
  40 K = 53 %) kann direkt ins Netz, *kalt* (Rücklauf bis 10 °C, 35 K = 47 %) nur über die Wärmepumpe.
  Beladen füllt beide Teile immer im Verhältnis 53 : 47. Verlust je Teil = 4·10⁻⁵ × Inhalt je Stunde.
- **Load**: die Last.

Hier zur Anschauung ein Netz mit **festen** Größen für zwei Januarwochen. Auch WP (1 MW) und PtH
(1,6 MW) legen wir fest – sonst würde die Optimierung für nur zwei Wochen Betrieb keine WP bauen, weil
deren Kosten als Jahreskosten angesetzt sind.


In [ ]:
kurz = ts.loc["2022-01-10":"2022-01-23"]
def feste_leistungen(n):
    n.links.loc["wp_out", ["p_nom", "p_nom_extendable"]] = [1.0, False]
    n.links.loc["pth", ["p_nom", "p_nom_extendable"]] = [1.6 / 0.99, False]

n_kurz = network.build_network(cfg, kurz, A_m2=2650, V_m3=5000)
feste_leistungen(n_kurz)
display(n_kurz.generators[["bus", "p_nom", "p_nom_extendable", "capital_cost"]])
display(n_kurz.links[["bus0", "bus1", "bus2", "efficiency", "efficiency2", "p_nom_extendable"]])
display(n_kurz.stores[["bus", "e_nom", "standing_loss", "e_cyclic"]])

## 3 Vier Zusatzbedingungen

Alles andere (Energiebilanz je Knoten, Grenzen, Speicherbilanz) erledigt PyPSA. Wir ergänzen nur:

1. **Fläche:** Kollektorfeld × 2,3 (Reihenabstand, Technik) + Speichervolumen / 2,65 m³/m² ≤ 13.000 m².
2. **Speicher wächst im festen Verhältnis:** der kalte Teil ist immer 0,875 × der heiße Teil (35/40),
   weil Wasser von 10 auf 85 °C erwärmt wird. So gilt die Aufteilung heiß/kalt in **jeder Stunde**, nicht nur
   als Jahresbilanz.
3. **Luft-WP bei Kälte:** Ihre Wärmeleistung sinkt von 100 % (ab −1 °C) linear auf 80 % bei −7 °C
   (darunter ist sie aus); die gebaute WP-Leistung gilt als Bezug. Die Speicher-WP ist nicht betroffen.
4. **Temperaturkennlinie des Speichers** (kein Teil von `extra_functionality`, sondern in `network.optimize`):
   Heißes Wasser liegt oben, darunter eine Übergangszone (≈ 30 % des Volumens, Dronninglund). Direkt ins Netz
   (80 °C + 5 K) geht es nur, wenn die **heiße Schicht** dicker ist als diese Zone; Wasser mit 45 °C im kalten
   Teil hebt die Temperatur oben nicht. Weil eine solche Schwelle im LP nicht linear ist, wird gerechnet, dann
   werden Stunden mit zu dünner heißer Schicht für die Direktentladung gesperrt und neu gerechnet (bis keine
   Verstöße mehr da sind). Darunter holt die Speicher-WP die Wärme heraus (`umlagern` bucht heiß → kalt).
   Parameter `ptes.schicht_min` (0 = aus). Die Kennlinie liefert auch die Speichertemperaturen in den Grafiken.


In [ ]:
import inspect, serts.constraints as c
print(inspect.getsource(c.extra_functionality))

## 4 Optimieren

`network.optimize` ruft PyPSA mit dem Solver HiGHS auf. Zielfunktion = Summe aus
Strom × Strompreis (jede Stunde) + Jahreskosten der Leistungen, die optimiert werden (hier keine –
alles ist fest vorgegeben, es wird nur der Betrieb optimiert).

In [ ]:
network.optimize(n_kurz, cfg, kurz)
print(f"Kosten im Zeitraum (Strom + Speicher-Tie-Breaker): {n_kurz.objective:,.0f} €")
k_kurz, r_kurz = kpi.auswerten(n_kurz, cfg, kurz)

fig, ax = plt.subplots(figsize=(11, 4))
ergebnisplots._stapel(ax, ergebnisplots._quellen(r_kurz))
ax.plot(r_kurz.index, r_kurz["last"], "k", label="Last")
ax.legend(ncol=6, fontsize=8); ax.set_ylabel("MW"); ax.set_title("Wer deckt die Last? (2 Januarwochen)")
plt.show()

## 5 Kosten nach VDI 2067 und Kennzahlen

Jede Investition A0 wird in **Jahreskosten** umgerechnet (Betrachtungszeitraum T = 20 a, Zins 5 %):

* kapitalgebunden: (A0 + Ersatzbeschaffungen − Restwert) × Annuitätsfaktor,
* betriebsgebunden: A0 × O&M-Satz,
* bedarfsgebunden: Strom × Preis (bei realer Rechnung ohne Preisänderung).

LCOH = Jahreskosten / Wärmemenge. Beispiel: Kollektor mit 25 a Lebensdauer hat nach 20 a noch einen
Restwert, eine Komponente mit 15 a wird einmal ersetzt.

Auf jede Investition kommen **10 % Nebenkosten** (Planung etc.). Die BEW-Förderquote gilt nur für Komponenten,
die als förderfähig markiert sind (`oekonomie.foerderfaehig`). Das **Wärmenetz** (Trasse, Hausanschlüsse inkl. Übergabestationen) wird getrennt ausgewiesen; der Strompreis
besteht intern aus Energie + Netzentgelt + Abgaben (für die Zeitreihe), erscheint in den Ergebnissen aber als ein Block.


In [ ]:
for TN in (15, 20, 25):
    print(f"Lebensdauer {TN} a -> Jahreskosten je 1 € Invest: {economics.kapitalfaktor_vdi(0.05, 20, TN):.4f}")
nk = economics.netzkosten(cfg)
print(f"Verteilnetz: Trasse {nk['netz_invest_trasse_EUR']/1e6:.2f} Mio €, "
      f"Hausanschlüsse {nk['netz_invest_hausanschluss_EUR']/1e6:.2f} Mio €, "
      f"{nk['kosten_a_netz_system']/1e3:.0f} T€/a")

## 6 Der eigentliche Lauf: Größen optimieren (≈ 1–2 min)

Jetzt werden Kollektor, Speicher, WP und PtH **mitoptimiert**. Weil große Erdbeckenspeicher je m³
billiger sind, rechnet `loese` fünf Läufe – einen je Kostensegment der (stetigen, mit der Größe
degressiven) Speicherkostenkurve – und nimmt den günstigsten. Neu in den Kennzahlen: `speicher_charakter` (saisonal / gemischt / kurzzeit) und der
gewichtete Strompreis.

In [ ]:
n, k, r = loese(cfg, ts)
for zeile in k["segmente_vergleich"]:
    print("Kostensegment", zeile)
pd.Series({
    "Kollektor [m²]": k["A_koll_m2"], "Speicher [m³]": k["V_m3"], "WP [MW]": k["P_WP_MW"],
    "PtH [MW]": k["P_PtH_MW"], "LCOH Erzeugung [€/MWh]": k["lcoh"],
    "Wärmepreis inkl. Netz [€/MWh]": k.get("lcoh_inkl_netz"), "Gas-Referenz inkl. Netz": k.get("gas_lcoh_inkl_netz"),
    "Solaranteil": k["solar_gesamt"], "CO₂ [t/a]": k["co2_t_a"],
}).round(2)

In [ ]:
import tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp())
for bild in ergebnisplots.alle(r, k, tmp):
    display(Image(filename=str(bild)))

## 7 Strom: fest oder Day-Ahead? – zwei Läufe vergleichen

Der Strompreis ist der größte Hebel. Das Modell kann ihn als **fester Tarif** (Basis, 30 ct/kWh) oder als
**Day-Ahead-Zeitreihe** rechnen (`strom.tarif`):

```
Preis(t) = Day-Ahead(t) + Vertriebsmarge (25 €/MWh) + Netzentgelt (93 €/MWh) + Abgaben (57 €/MWh)
```

* Day-Ahead-Stundenpreise DE-LU: Bundesnetzagentur | SMARD.de (CC BY 4.0), über die Energy-Charts-API geladen
  (`uv run serts fetch-strompreise 2022 2023 2024`). Jahresmittel: 2022 = 235 (Krise), 2023 = 95, 2024 = 78,5 €/MWh.
* Das Preisjahr ist vom Wetterjahr unabhängig (`strom.preisjahr`). Netzentgelt und Abgaben sind konstant,
  ein **Leistungspreis** ist nicht modelliert.
* Zwei Vergleiche über Szenario-Dateien: `strom_day_ahead_2024` (andere Preishöhe **und** -struktur) und
  `strom_day_ahead_normiert` (gleicher Jahresmittelwert wie der feste Tarif → zeigt nur den Effekt der
  zeitlichen Struktur).

In [ ]:
ZWEITER_LAUF = "strom_day_ahead_2024"      # oder "strom_day_ahead_normiert", "strom_day_ahead_2022"
cfg2 = load_config(ZWEITER_LAUF)
ts2, _ = build_zeitreihen(cfg2)
n2, k2, r2 = loese(cfg2, ts2)

zeilen = {
    "Strompreis Jahresmittel [€/MWh]": "strompreis_mittel_EUR_MWh",
    "Strompreis gewichtet bezahlt [€/MWh]": "strompreis_gewichtet_EUR_MWh",
    "Kollektor [m²]": "A_koll_m2", "Speicher [m³]": "V_m3", "WP [MW]": "P_WP_MW",
    "Strombezug [MWh/a]": "E_strom_MWh", "Solaranteil": "solar_gesamt",
    "LCOH Erzeugung [€/MWh]": "lcoh", "Wärmepreis inkl. Netz [€/MWh]": "lcoh_inkl_netz",
    "Gas-Fernwärme inkl. Netz [€/MWh]": "gas_lcoh_inkl_netz",
}
tab = pd.DataFrame({"Basis (fest)": {z: k.get(key) for z, key in zeilen.items()},
                    ZWEITER_LAUF: {z: k2.get(key) for z, key in zeilen.items()}})
tab.round(2)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
fenster = slice("2022-01-10", "2022-01-23")
ax[0].plot(ts.loc[fenster, "strompreis"], label="fest", lw=1.2)
ax[0].plot(ts2.loc[fenster, "strompreis"], label=ZWEITER_LAUF, lw=1.2)
ax[0].set_ylabel("Strompreis [€/MWh]"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(r.loc[fenster, "strom_netz"], label="Strombezug fest", lw=1)
ax[1].plot(r2.loc[fenster, "strom_netz"], label=f"Strombezug {ZWEITER_LAUF}", lw=1)
ax[1].set_ylabel("Strombezug [MW]"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[0].set_title("Reagiert der Betrieb auf die Preise? (zwei Januarwochen)")
plt.tight_layout()
plt.show()

## 8 Wann wird der Speicher saisonal?

Die Kennzahlen `speicher_zyklen` und `speicher_winteranteil` zeigen, wie der Speicher genutzt wird:
**saisonal** = unter 2 Zyklen pro Jahr und mindestens 60 % der Entladung von November bis März.
Im Basislauf ist er ein Kurzzeit-/Wochenspeicher (≈ 5 Zyklen). Warum, und was sich ändern müsste, steht in
`docs/saisonaler_speicher.md`. Hier das einfachste Beispiel: **Speicherkostenkurve × 0,35** (Szenario
`saisonal_demo`), alles andere wie Basis.

In [ ]:
cfg3 = load_config("saisonal_demo")
ts3, _ = build_zeitreihen(cfg3)
n3, k3, r3 = loese(cfg3, ts3)

vergleich = pd.DataFrame({
    "Basis": {x: k.get(x) for x in ("A_koll_m2", "V_m3", "speicher_zyklen", "speicher_winteranteil", "speicher_charakter", "solar_gesamt", "lcoh")},
    "Speicher x0,35": {x: k3.get(x) for x in ("A_koll_m2", "V_m3", "speicher_zyklen", "speicher_winteranteil", "speicher_charakter", "solar_gesamt", "lcoh")},
})
display(vergleich)

fig, ax = plt.subplots(figsize=(11, 3.5))
r["speicherinhalt"].resample("D").mean().plot(ax=ax, label="Basis")
r3["speicherinhalt"].resample("D").mean().plot(ax=ax, label="Speicher x0,35")
ax.set_ylabel("Speicherinhalt [MWh]"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Kurzzeit- gegen Saisonspeicher: Inhalt über das Jahr")
plt.show()

## 9 Annahmen ändern

* **dauerhaft**: Wert in `config/base.yaml` ändern (mit Quelle), danach `uv run serts annahmen`;
* **als Variante**: kleine YAML-Datei in `config/scenarios/` (nur Abweichungen), z. B. `rl_50.yaml`,
  `strom_day_ahead_2024.yaml`, `saisonal_demo.yaml`; Aufruf `uv run serts run --szenario <name>`;
* **im Notebook**: `cfg.variante("pfad", wert)` erzeugt eine geänderte Kopie.

In [ ]:
teurer_strom = cfg.variante("strom.preis_fest", 400)
ts2, _ = build_zeitreihen(teurer_strom)
kurz2 = ts2.loc["2022-01-10":"2022-01-23"]
n2 = network.build_network(teurer_strom, kurz2, A_m2=2650, V_m3=5000)
feste_leistungen(n2)
network.optimize(n2, teurer_strom, kurz2)
print(f"2 Januarwochen: {n2.objective:,.0f} € mit 40 ct/kWh statt {n_kurz.objective:,.0f} € mit 30 ct/kWh")